<a href="https://colab.research.google.com/github/nhjung-phd/TimeSeriesAnalysis/blob/main/practice/Student_TimeSeries_Forecasting_Example_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 금 가격 시계열 분석과 예측 v5

- 과제 주제: 주식이 아닌 시계열 데이터를 활용한 예측 예제
- 데이터: `gold_historical_data.csv`
- 목표: 금 가격의 추세와 변동성을 분석하고, `Naive`, `Moving Average`, `AR`, `MA`, `ARIMA` 모델의 예측 성능을 비교한다.

이 노트북은 `EDA → 시계열 특성 파악 → baseline 예측 → AR/MA/ARIMA 비교 → 결과 기반 해석` 순서로 진행한다.


## Colab 파일 업로드 안내

이 노트북은 `gold_historical_data.csv` 파일을 Colab에 직접 업로드한 뒤 실행하는 버전이다.
아래 코드 셀을 먼저 실행한 뒤, 로컬에 있는 CSV 파일을 선택하면 된다.


In [ ]:
# Colab에서 로컬 CSV 파일을 직접 업로드한다.
# 이 셀을 실행하면 파일 선택 창이 뜨고, 로컬 컴퓨터의 CSV 파일을 업로드할 수 있다.
from google.colab import files

# 업로드 결과는 딕셔너리 형태로 저장되며, key가 업로드된 파일명이다.
uploaded = files.upload()

# 업로드가 정상적으로 되었는지 파일명을 바로 출력해서 확인한다.
print('업로드된 파일:', list(uploaded.keys()))


### 업로드 후 확인 사항

- 업로드된 파일명이 원래 이름과 다르더라도, 아래 데이터 로드 셀에서 자동으로 첫 번째 업로드 파일을 읽는다.
- Colab 런타임이 다시 시작되면 파일을 다시 업로드해야 한다.


## 1. 분석 개요

이 노트북에서 확인할 핵심 질문은 다음과 같다.

1. 금 가격 시계열은 장기적으로 어떤 추세를 보이는가?
2. 단기 급등락과 변동성 확대 구간은 언제 나타나는가?
3. 단순 baseline과 AR, MA, ARIMA 계열 모델을 비교하면 어떤 차이가 있는가?
4. 실제 결과를 바탕으로 어떤 모델이 더 적절했는가?


In [ ]:
# 필요한 라이브러리를 불러온다.
# warnings는 불필요한 경고 메시지를 숨겨 노트북 출력이 지저분해지는 것을 막는다.
import warnings
warnings.filterwarnings('ignore')

# 수치 계산과 데이터 처리를 위한 기본 라이브러리이다.
import numpy as np
import pandas as pd

# 시각화를 위한 라이브러리이다.
import matplotlib.pyplot as plt
import seaborn as sns

# 예측 성능 평가를 위한 지표 함수이다.
from sklearn.metrics import mean_absolute_error, mean_squared_error

# AR 모델과 ARIMA 모델 구현을 위해 statsmodels를 사용한다.
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA

# 그래프 스타일과 숫자 표시 형식을 보기 좋게 설정한다.
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


## 2. 데이터 로드 및 기본 점검

시계열 분석에서는 모델링 전에 데이터 상태를 먼저 확인해야 한다.
이 단계에서는 날짜형 변환, 정렬, 결측치, 중복 날짜, 관측 기간을 점검한다.


In [ ]:
# 업로드된 파일이 있으면 그 파일명을 사용하고, 없으면 기본 파일명을 사용한다.
# Colab에서는 같은 파일을 여러 번 올릴 경우 파일명 뒤에 (1), (2)가 붙을 수 있으므로 이 처리가 필요하다.
file_name = list(uploaded.keys())[0] if 'uploaded' in globals() and len(uploaded) > 0 else 'gold_historical_data.csv'
print('사용할 파일명:', file_name)

# CSV 파일을 읽는다.
df = pd.read_csv(file_name)

# 날짜 컬럼을 datetime 형식으로 바꿔야 시계열 분석이 가능하다.
df['Date'] = pd.to_datetime(df['Date'])

# 날짜 순서가 섞여 있으면 시계열 분할과 예측 결과가 왜곡될 수 있으므로 오름차순 정렬한다.
df = df.sort_values('Date').reset_index(drop=True)

# Date를 인덱스로 설정한 시계열 전용 데이터프레임을 만든다.
gold = df.set_index('Date').copy()

# 데이터의 기본 구조를 먼저 점검한다.
print('데이터 크기:', gold.shape)
print('시작일:', gold.index.min().date())
print('종료일:', gold.index.max().date())
print('중복 날짜 수:', gold.index.duplicated().sum())
print('\n결측치 개수:')
print(gold.isna().sum())

# 실제 데이터 형태를 눈으로 확인하기 위해 상위 5개 행을 출력한다.
print('\n상위 5개 행:')
display(gold.head())

# 각 변수의 규모와 분포를 파악하기 위해 기초 통계량을 확인한다.
print('\n기초 통계량:')
display(gold.describe().T)


### 데이터 점검 해석

- 날짜 중복과 결측치가 없으면 기본적인 시계열 분석을 진행하기 좋다.
- `Close`를 예측 대상으로 사용하고, `Open`, `High`, `Low`, `Volume`은 보조 해석 변수로 활용할 수 있다.
- 금융 시계열은 이상치와 급격한 변동이 존재할 수 있으므로 이후 EDA에서 이를 확인한다.


## 3. 탐색적 데이터 분석(EDA)

이 단계에서는 전체 추세, 이동평균, 거래량, 일간 수익률 분포를 시각화하여 시계열의 성격을 이해한다.


In [ ]:
# 분석에 필요한 파생 변수를 만든다.
# Return: 전일 대비 수익률
# MA20/60/120: 20일, 60일, 120일 이동평균
# Volatility20: 최근 20일 수익률 표준편차
# 이 변수들은 추세와 변동성을 함께 보기 위해 사용한다.
gold['Return'] = gold['Close'].pct_change()
gold['MA20'] = gold['Close'].rolling(window=20).mean()
gold['MA60'] = gold['Close'].rolling(window=60).mean()
gold['MA120'] = gold['Close'].rolling(window=120).mean()
gold['Volatility20'] = gold['Return'].rolling(window=20).std()

# 한 화면에서 주요 EDA 그래프를 4개 배치한다.
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1) 종가 시계열: 전체적인 장기 추세를 확인한다.
axes[0, 0].plot(gold.index, gold['Close'], label='Close', linewidth=1.5)
axes[0, 0].set_title('Gold Close Price Over Time')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Price')
axes[0, 0].legend()

# 2) 이동평균선 비교: 단기 변동을 줄이고 추세를 더 쉽게 확인한다.
axes[0, 1].plot(gold.index, gold['Close'], label='Close', alpha=0.6)
axes[0, 1].plot(gold.index, gold['MA20'], label='MA20')
axes[0, 1].plot(gold.index, gold['MA60'], label='MA60')
axes[0, 1].plot(gold.index, gold['MA120'], label='MA120')
axes[0, 1].set_title('Close Price with Moving Averages')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Price')
axes[0, 1].legend()

# 3) 거래량 추이: 특정 시점에 거래량 급증이 있는지 확인한다.
axes[1, 0].plot(gold.index, gold['Volume'], color='tab:orange')
axes[1, 0].set_title('Trading Volume Over Time')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Volume')

# 4) 일간 수익률 분포: 수익률이 대칭적인지, 극단값이 많은지 확인한다.
axes[1, 1].hist(gold['Return'].dropna(), bins=50, color='tab:green', edgecolor='black', alpha=0.8)
axes[1, 1].set_title('Distribution of Daily Returns')
axes[1, 1].set_xlabel('Daily Return')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


In [ ]:
# 수익률 시계열과 rolling volatility를 별도로 그린다.
# 첫 번째 그래프는 일간 변동 방향을 보여주고,
# 두 번째 그래프는 변동성이 커지는 구간을 확인하는 데 사용한다.
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# 일간 수익률 시계열
axes[0].plot(gold.index, gold['Return'], color='tab:purple', linewidth=1)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Daily Returns Over Time')
axes[0].set_ylabel('Return')

# 20일 rolling volatility
axes[1].plot(gold.index, gold['Volatility20'], color='tab:red', linewidth=1.5)
axes[1].set_title('20-Day Rolling Volatility')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Volatility')

plt.tight_layout()
plt.show()

# 가장 큰 하락 날짜와 가장 큰 상승 날짜를 출력해 이상 변동 구간을 확인한다.
print('일간 수익률 하위 5개 날짜')
display(gold['Return'].nsmallest(5).to_frame('Return'))

print('일간 수익률 상위 5개 날짜')
display(gold['Return'].nlargest(5).to_frame('Return'))


### EDA 해석 포인트

- 이동평균선은 단기 변동을 완화해 장기 추세를 더 쉽게 보여준다.
- 일간 수익률은 가격 자체보다 더 직접적으로 변동성을 보여준다.
- rolling volatility가 높아지는 구간은 예측 난이도가 커질 가능성이 있다.
- 급등락 날짜를 확인하면 모델 성능이 낮아지는 구간을 해석하는 데 도움이 된다.


## 4. 예측용 데이터 준비

이제 `Close`를 예측 대상으로 두고 학습용과 테스트용 데이터를 시간 순서에 따라 분리한다.
시계열 예측에서는 데이터를 무작위로 섞지 않는 것이 중요하다.


In [ ]:
# 예측 대상은 종가(Close)이다.
# 여기서는 마지막 20% 구간을 테스트 데이터로 사용한다.
# 시계열 데이터는 순서를 유지해야 하므로 무작위 셔플을 하지 않는다.
series = gold['Close'].copy()
train_size = int(len(series) * 0.8)
train = series.iloc[:train_size]
test = series.iloc[train_size:]

print('학습 데이터 개수:', len(train))
print('테스트 데이터 개수:', len(test))
print('학습 시작일:', train.index.min().date(), '| 학습 종료일:', train.index.max().date())
print('테스트 시작일:', test.index.min().date(), '| 테스트 종료일:', test.index.max().date())

# 공통 평가 함수
# actual과 predicted를 날짜 인덱스로 맞춘 뒤, MAE와 RMSE를 계산한다.
def evaluate_forecast(actual, predicted):
    aligned = pd.concat([actual, predicted], axis=1).dropna()
    aligned.columns = ['actual', 'predicted']
    mae = mean_absolute_error(aligned['actual'], aligned['predicted'])
    rmse = np.sqrt(mean_squared_error(aligned['actual'], aligned['predicted']))
    return mae, rmse


## 5. Naive Baseline 예측

가장 단순한 기준선은 다음 시점의 값을 직전 시점 값으로 예측하는 방법이다.
금융 시계열에서는 이 단순한 방법이 생각보다 강한 baseline이 될 수 있다.


In [ ]:
# Naive baseline
# 가장 단순한 방법으로, 직전 시점 값을 다음 시점의 예측값으로 사용한다.
naive_pred = test.shift(1)

# 테스트 구간 첫 값은 이전 테스트 값이 없으므로 학습 데이터 마지막 값을 사용한다.
naive_pred.iloc[0] = train.iloc[-1]

# 성능 지표를 계산한다.
naive_mae, naive_rmse = evaluate_forecast(test, naive_pred)
print(f'Naive MAE : {naive_mae:,.4f}')
print(f'Naive RMSE: {naive_rmse:,.4f}')

# 최근 학습 구간과 테스트 구간을 함께 그려 예측이 어떻게 따라가는지 확인한다.
plt.figure(figsize=(16, 5))
plt.plot(train.index[-120:], train.tail(120), label='Train (last 120)')
plt.plot(test.index, test, label='Actual Test', linewidth=2)
plt.plot(test.index, naive_pred, label='Naive Forecast', linestyle='--')
plt.title('Naive Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()


## 6. Moving Average Baseline 예측

최근 며칠간의 평균을 사용해 다음 값을 예측하면 노이즈를 일부 완화할 수 있다.
여기서는 5일, 20일 이동평균 기반 예측을 비교한다.


In [ ]:
# 최근 window일 평균을 다음 시점 예측값으로 사용하는 함수이다.
# 테스트 구간에서 실제값이 하나씩 공개된다고 가정하고 history를 계속 갱신한다.
def moving_average_forecast(train_series, test_series, window):
    history = list(train_series.values)
    predictions = []

    for actual_value in test_series.values:
        # 가장 최근 window개 관측값의 평균을 사용한다.
        pred = np.mean(history[-window:])
        predictions.append(pred)

        # 실제 관측값을 history에 추가해 다음 시점 예측에 반영한다.
        history.append(actual_value)

    return pd.Series(predictions, index=test_series.index)

# 5일, 20일 이동평균 예측을 각각 계산한다.
ma5_pred = moving_average_forecast(train, test, window=5)
ma20_pred = moving_average_forecast(train, test, window=20)

# 성능을 각각 평가한다.
ma5_mae, ma5_rmse = evaluate_forecast(test, ma5_pred)
ma20_mae, ma20_rmse = evaluate_forecast(test, ma20_pred)

print(f'MA(5)  MAE : {ma5_mae:,.4f}')
print(f'MA(5)  RMSE: {ma5_rmse:,.4f}')
print(f'MA(20) MAE : {ma20_mae:,.4f}')
print(f'MA(20) RMSE: {ma20_rmse:,.4f}')

# 두 이동평균 예측을 실제값과 함께 시각화한다.
plt.figure(figsize=(16, 5))
plt.plot(test.index, test, label='Actual Test', linewidth=2)
plt.plot(test.index, ma5_pred, label='MA(5) Forecast', linestyle='--')
plt.plot(test.index, ma20_pred, label='MA(20) Forecast', linestyle=':')
plt.title('Moving Average Forecasts vs Actual')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()


## 7. AR 모델 예측

AR(Autoregressive) 모델은 과거 관측값들을 사용해 현재 값을 예측한다.
여기서는 `AutoReg(lags=5)`를 사용하여 최근 5개 시점 정보를 반영한다.


In [ ]:
# AR(5) 모델
# 과거 5개 시점의 종가를 사용해 현재 값을 설명하는 자기회귀 모델이다.
ar_model = AutoReg(train, lags=5, old_names=False)
ar_result = ar_model.fit()

# 테스트 구간 길이만큼 예측한다.
ar_forecast_values = ar_result.predict(start=len(train), end=len(train) + len(test) - 1)

# 예측값에 테스트 날짜 인덱스를 붙여 실제값과 바로 비교할 수 있게 만든다.
ar_pred = pd.Series(np.asarray(ar_forecast_values), index=test.index)

# 성능을 평가한다.
ar_mae, ar_rmse = evaluate_forecast(test, ar_pred)
print(f'AR(5) MAE : {ar_mae:,.4f}')
print(f'AR(5) RMSE: {ar_rmse:,.4f}')

# 실제값과 AR 예측값을 비교한다.
plt.figure(figsize=(16, 5))
plt.plot(test.index, test, label='Actual Test', linewidth=2)
plt.plot(test.index, ar_pred, label='AR(5) Forecast', linestyle='--')
plt.title('AR(5) Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()


## 8. MA 모델 예측

`statsmodels`에서는 순수 MA(q) 모델을 보통 ARIMA 형태로 표현한다.
여기서는 차분을 포함한 `ARIMA(0,1,5)`를 사용해 MA 성분 중심의 예측을 수행한다.


In [ ]:
# MA 성분 중심 모델
# statsmodels에서는 순수 MA(q)를 보통 ARIMA(0, d, q) 형태로 구현한다.
# 여기서는 1차 차분과 5차 MA 성분을 가진 ARIMA(0,1,5)를 사용한다.
ma_model = ARIMA(train, order=(0, 1, 5))
ma_result = ma_model.fit()

# forecast 결과는 numpy 배열로 꺼낸 뒤 테스트 인덱스를 부여한다.
ma_forecast_values = ma_result.forecast(steps=len(test)).to_numpy()
ma_pred = pd.Series(ma_forecast_values, index=test.index)

# 성능을 평가한다.
ma_model_mae, ma_model_rmse = evaluate_forecast(test, ma_pred)
print(f'MA-based ARIMA(0,1,5) MAE : {ma_model_mae:,.4f}')
print(f'MA-based ARIMA(0,1,5) RMSE: {ma_model_rmse:,.4f}')

# 실제값과 예측값을 시각화한다.
plt.figure(figsize=(16, 5))
plt.plot(test.index, test, label='Actual Test', linewidth=2)
plt.plot(test.index, ma_pred, label='MA-based Forecast', linestyle='--')
plt.title('MA-based ARIMA(0,1,5) Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()


## 9. ARIMA 모델 예측

ARIMA는 자기회귀(AR), 차분(I), 이동평균(MA)을 결합한 대표적인 고전 시계열 모델이다.
여기서는 AR과 MA를 함께 포함하는 `ARIMA(2,1,2)`를 사용해 통합 모델 성능을 확인한다.


In [ ]:
# ARIMA(2,1,2) 모델
# ARIMA는 AR 성분, 차분 성분, MA 성분을 함께 포함하는 통합 시계열 모델이다.
# 여기서는 AR 2차, 1차 차분, MA 2차 구조를 가진 모델을 사용한다.
arima_model = ARIMA(train, order=(2, 1, 2))
arima_result = arima_model.fit()

# 테스트 구간 길이만큼 예측하고, 결과에 테스트 날짜 인덱스를 붙인다.
arima_forecast_values = arima_result.forecast(steps=len(test)).to_numpy()
arima_pred = pd.Series(arima_forecast_values, index=test.index)

# 성능을 평가한다.
arima_mae, arima_rmse = evaluate_forecast(test, arima_pred)
print(f'ARIMA(2,1,2) MAE : {arima_mae:,.4f}')
print(f'ARIMA(2,1,2) RMSE: {arima_rmse:,.4f}')

# 최근 학습 구간과 테스트 구간을 함께 시각화해 추세를 얼마나 따라가는지 본다.
plt.figure(figsize=(16, 5))
plt.plot(train.index[-120:], train.tail(120), label='Train (last 120)')
plt.plot(test.index, test, label='Actual Test', linewidth=2)
plt.plot(test.index, arima_pred, label='ARIMA(2,1,2) Forecast', linestyle='--')
plt.title('ARIMA(2,1,2) Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()


## 10. 모델 성능 비교

이제 각 모델의 성능을 한 표에서 비교한다.
핵심은 복잡한 모델이 항상 좋은 것이 아니라, baseline 대비 실제로 얼마나 개선되었는지 확인하는 것이다.


In [ ]:
# 앞에서 계산한 모든 모델의 성능을 하나의 표로 정리한다.
# RMSE 기준 오름차순 정렬을 사용해 어떤 모델이 더 좋은지 바로 확인할 수 있게 한다.
results = pd.DataFrame({
    'Model': [
        'Naive',
        'Moving Average (5)',
        'Moving Average (20)',
        'AR(5)',
        'MA-based ARIMA(0,1,5)',
        'ARIMA(2,1,2)'
    ],
    'MAE': [
        naive_mae,
        ma5_mae,
        ma20_mae,
        ar_mae,
        ma_model_mae,
        arima_mae
    ],
    'RMSE': [
        naive_rmse,
        ma5_rmse,
        ma20_rmse,
        ar_rmse,
        ma_model_rmse,
        arima_rmse
    ]
}).sort_values('RMSE').reset_index(drop=True)

# 성능 비교 표를 출력한다.
display(results)

# 핵심 모델들의 예측선을 한 그래프에 겹쳐 비교한다.
plt.figure(figsize=(16, 6))
plt.plot(test.index, test, label='Actual Test', linewidth=2, color='black')
plt.plot(test.index, naive_pred, label='Naive', linestyle='--')
plt.plot(test.index, ar_pred, label='AR(5)', linestyle=':')
plt.plot(test.index, ma_pred, label='MA-based', linestyle='-.')
plt.plot(test.index, arima_pred, label='ARIMA(2,1,2)', linewidth=2)
plt.title('Forecast Comparison on Test Period')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()


## 11. Walk-Forward 1-Step 비교

앞선 비교는 학습 구간으로 한 번만 모델을 적합한 뒤 테스트 구간 전체를 예측한 결과였다.
하지만 시계열 예측에서는 매 시점마다 새 관측치를 반영해 한 스텝씩 예측하는 walk-forward 평가가 더 현실적일 수 있다.

여기서는 모든 모델을 동일한 `1-step ahead` 방식으로 비교한다.
다만 ARIMA 계열 모델을 반복 적합해야 하므로 Colab 실행 시간을 고려해 테스트 구간의 일부(`WF_STEPS`)만 사용한다.


In [ ]:
# Walk-forward 1-step 비교를 위한 평가 구간을 설정한다.
# ARIMA 계열 모델은 매 스텝마다 다시 적합해야 하므로 계산량이 크다.
# 따라서 Colab 실행 시간을 고려해 테스트 구간 앞부분 일부만 사용한다.
WF_STEPS = min(60, len(test))
wf_test = test.iloc[:WF_STEPS]

print('walk-forward 평가 스텝 수:', WF_STEPS)
print('평가 시작일:', wf_test.index.min().date())
print('평가 종료일:', wf_test.index.max().date())


In [ ]:
# 모든 모델을 동일한 1-step walk-forward 방식으로 평가한다.
# 핵심 아이디어는 "한 시점 예측 후 실제값을 반영하고 다음 시점을 다시 예측"하는 것이다.
# 이렇게 하면 모든 모델이 동일한 정보 조건에서 비교된다.
wf_history = list(train.values)
wf_predictions = {
    'Naive': [],
    'Moving Average (5)': [],
    'Moving Average (20)': [],
    'AR(5)': [],
    'MA-based ARIMA(0,1,5)': [],
    'ARIMA(2,1,2)': []
}

for actual_value in wf_test.values:
    # 현재 시점까지의 history를 pandas Series로 변환한다.
    # statsmodels 모델은 Series 입력이 더 안정적이다.
    history_series = pd.Series(wf_history)

    # 1) Naive: 직전 값 사용
    wf_predictions['Naive'].append(wf_history[-1])

    # 2) Moving Average: 최근 관측값 평균 사용
    wf_predictions['Moving Average (5)'].append(np.mean(wf_history[-5:]))
    wf_predictions['Moving Average (20)'].append(np.mean(wf_history[-20:]))

    # 3) AR(5): 매 스텝마다 다시 적합해 1-step 예측
    ar_wf_model = AutoReg(history_series, lags=5, old_names=False)
    ar_wf_result = ar_wf_model.fit()
    ar_wf_forecast = ar_wf_result.predict(start=len(history_series), end=len(history_series)).iloc[0]
    wf_predictions['AR(5)'].append(ar_wf_forecast)

    # 4) MA-based ARIMA(0,1,5): MA 성분 중심 모델의 1-step 예측
    ma_wf_model = ARIMA(history_series, order=(0, 1, 5))
    ma_wf_result = ma_wf_model.fit()
    ma_wf_forecast = ma_wf_result.forecast(steps=1).iloc[0]
    wf_predictions['MA-based ARIMA(0,1,5)'].append(ma_wf_forecast)

    # 5) ARIMA(2,1,2): 통합 모델의 1-step 예측
    arima_wf_model = ARIMA(history_series, order=(2, 1, 2))
    arima_wf_result = arima_wf_model.fit()
    arima_wf_forecast = arima_wf_result.forecast(steps=1).iloc[0]
    wf_predictions['ARIMA(2,1,2)'].append(arima_wf_forecast)

    # 실제 관측값을 history에 추가해 다음 시점 예측에 반영한다.
    wf_history.append(actual_value)

# 모델별 walk-forward 성능을 표로 정리한다.
wf_result_rows = []
for model_name, preds in wf_predictions.items():
    pred_series = pd.Series(preds, index=wf_test.index)
    mae, rmse = evaluate_forecast(wf_test, pred_series)
    wf_result_rows.append({
        'Model': model_name,
        'MAE': mae,
        'RMSE': rmse
    })

wf_results = pd.DataFrame(wf_result_rows).sort_values('RMSE').reset_index(drop=True)
display(wf_results)


In [ ]:
# Walk-forward 1-step 예측 결과를 시각화한다.
# 실제값과 여러 모델의 1-step 예측선을 같은 그래프에 그려 어떤 모델이 더 잘 따라가는지 본다.
plt.figure(figsize=(16, 6))
plt.plot(wf_test.index, wf_test, label='Actual', linewidth=2, color='black')
plt.plot(wf_test.index, wf_predictions['Naive'], label='Naive', linestyle='--')
plt.plot(wf_test.index, wf_predictions['AR(5)'], label='AR(5)', linestyle=':')
plt.plot(wf_test.index, wf_predictions['MA-based ARIMA(0,1,5)'], label='MA-based', linestyle='-.')
plt.plot(wf_test.index, wf_predictions['ARIMA(2,1,2)'], label='ARIMA(2,1,2)', linewidth=2)
plt.title('Walk-Forward 1-Step Forecast Comparison')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()


### Walk-Forward 비교 해석 포인트

- 이 비교는 모든 모델이 같은 시점 정보만 사용하도록 맞춘 더 공정한 평가이다.
- 테스트 구간 전체를 한 번에 예측하는 방식보다, 시계열 운영 환경에 더 가깝다.
- 만약 ARIMA 성능이 이 구간에서 개선된다면, 이전 결과에는 장기 horizon 누적 오차 영향이 있었을 가능성이 있다.
- 반대로 여전히 baseline보다 나쁘다면, 현재 데이터에서는 단순 모델이 더 적절할 수 있다.


## 12. 최종 요약

- 금 가격 데이터는 주식이 아닌 시계열 예측 예제로 활용하기 적절하다.
- EDA를 통해 추세, 변동성, 급등락 구간을 먼저 이해한 뒤 예측을 수행했다.
- Naive, Moving Average, AR, MA, ARIMA를 비교하여 baseline 대비 성능을 해석했다.
- 마지막 해석과 한계는 실제 실행 결과를 바탕으로 생성되도록 구성했다.
